# ENAPC-6G: Energy-Neutral Adaptive Communication Framework
## Full end-to-end implementation

This notebook implements the proposed pipeline:

1. Load and integrate DeepMIMO, 6G-PATH and 5G network-traffic data when local dataset files are supplied.
2. Preprocess channel, traffic and energy-related observations into a unified time-ordered feature table.
3. Train an LSTM energy-availability predictor.
4. Apply adaptive SWIPT power-splitting / time-switching control.
5. Optimize transmit power, bandwidth, harvesting allocation and scheduling with a multi-objective adaptive whale optimization algorithm (MOAWO).
6. Run an energy-neutral scheduling controller and evaluate throughput, energy consumption, delay, energy-neutrality ratio, packet delivery and fairness.
7. Compare baseline and proposed policies over channel, traffic-load and harvesting scenarios; export results and plots.

**Data note:** The three source datasets have different formats and may require dataset-specific extraction. Set the paths and column mappings in the configuration cell. If the files are not present, the notebook creates a reproducible simulation dataset so that every code cell can be executed and the complete algorithm can be tested. Results from this simulation are not measurements from the named public datasets. Replace the simulation data with the actual datasets before reporting dataset-based results.

## 1. Environment and imports

In [ ]:
# Colab: uncomment if required
# !pip -q install numpy pandas scipy scikit-learn matplotlib seaborn tensorflow openpyxl

import os, glob, json, math, random, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__)

## 2. Dataset paths and integration configuration

In [ ]:
# Set these to the folders/files in Google Drive or the local runtime.
# Example:
# from google.colab import drive
# drive.mount('/content/drive')
# DEEPMIMO_PATH = '/content/drive/MyDrive/ENAPC6G/DeepMIMO'
# PATH6G_PATH   = '/content/drive/MyDrive/ENAPC6G/6G-PATH'
# TRAFFIC_PATH  = '/content/drive/MyDrive/ENAPC6G/5G-Traffic'

DEEPMIMO_PATH = ""
PATH6G_PATH = ""
TRAFFIC_PATH = ""
OUTPUT_DIR = "/content/ENAPC6G_Results" if os.path.exists("/content") else "./ENAPC6G_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional column-name hints. The loader also uses automatic name matching.
COLUMN_HINTS = {
    "snr": ["snr", "sinr", "rsrp", "channel_gain", "path_gain", "gain"],
    "bandwidth": ["bandwidth", "bw", "prb", "resource_block"],
    "traffic": ["traffic", "packet", "bytes", "throughput", "load", "bitrate"],
    "delay": ["delay", "latency", "rtt"],
    "energy": ["energy", "battery", "harvest", "power"],
}

## 3. Load source files and normalize schemas

In [ ]:
def read_tabular_files(root):
    """Read supported tabular files recursively; returns concatenated rows."""
    if not root or not os.path.exists(root):
        return None
    if os.path.isfile(root):
        files = [root]
    else:
        patterns = ["**/*.csv", "**/*.parquet", "**/*.pq", "**/*.xlsx", "**/*.json"]
        files = []
        for pattern in patterns:
            files.extend(glob.glob(os.path.join(root, pattern), recursive=True))
    frames = []
    for fp in sorted(set(files)):
        try:
            ext = Path(fp).suffix.lower()
            if ext == ".csv": df = pd.read_csv(fp)
            elif ext in (".parquet", ".pq"): df = pd.read_parquet(fp)
            elif ext == ".xlsx": df = pd.read_excel(fp)
            elif ext == ".json": df = pd.read_json(fp, lines=True)
            else: continue
            if len(df):
                df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
                df["_source_file"] = Path(fp).name
                frames.append(df)
        except Exception as e:
            print("Skipped", fp, "->", str(e)[:120])
    return pd.concat(frames, ignore_index=True, sort=False) if frames else None

deepmimo_raw = read_tabular_files(DEEPMIMO_PATH)
path6g_raw = read_tabular_files(PATH6G_PATH)
traffic_raw = read_tabular_files(TRAFFIC_PATH)

print("DeepMIMO:", None if deepmimo_raw is None else deepmimo_raw.shape)
print("6G-PATH:", None if path6g_raw is None else path6g_raw.shape)
print("5G traffic:", None if traffic_raw is None else traffic_raw.shape)

## 4. Feature extraction and unified dataset

In [ ]:
def numeric_series(df, candidates, default=None):
    if df is not None:
        cols = list(df.columns)
        for hint in candidates:
            matched = [c for c in cols if hint in c.lower()]
            for c in matched:
                s = pd.to_numeric(df[c], errors="coerce")
                if s.notna().sum() > 0:
                    return s.reset_index(drop=True)
        # use first numeric column as a last-resort source feature
        nums = df.select_dtypes(include=np.number)
        if nums.shape[1]:
            return pd.to_numeric(nums.iloc[:, 0], errors="coerce").reset_index(drop=True)
    return None

def fit_length(series, n, default):
    if series is None or len(series) == 0:
        return np.full(n, default, dtype=float)
    x = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    x = x.interpolate(limit_direction="both").fillna(default).to_numpy(dtype=float)
    if len(x) >= n: return x[:n]
    return np.resize(x, n)

def robust_unit(x):
    x = np.asarray(x, dtype=float)
    lo, hi = np.nanpercentile(x, 2), np.nanpercentile(x, 98)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(x)
    return np.clip((x-lo)/(hi-lo), 0, 1)

# Use source-specific features where available; align records by row index after extraction.
source_lengths = [len(x) for x in [deepmimo_raw, path6g_raw, traffic_raw] if x is not None]
N = max(source_lengths) if source_lengths else 12000

snr_raw = fit_length(numeric_series(deepmimo_raw, COLUMN_HINTS["snr"]), N, 12.0)
path_gain_raw = fit_length(numeric_series(path6g_raw, COLUMN_HINTS["snr"]), N, 8.0)
traffic_raw_s = fit_length(numeric_series(traffic_raw, COLUMN_HINTS["traffic"]), N, 500.0)
delay_raw = fit_length(numeric_series(traffic_raw, COLUMN_HINTS["delay"]), N, 10.0)
energy_raw = fit_length(numeric_series(traffic_raw, COLUMN_HINTS["energy"]), N, 0.5)

# If no input files were supplied, generate a reproducible network trace.
USING_SIMULATED_INPUT = not any(x is not None for x in [deepmimo_raw, path6g_raw, traffic_raw])
rng = np.random.default_rng(SEED)
t = np.arange(N)
if USING_SIMULATED_INPUT:
    snr_raw = 10 + 5*np.sin(t/170) + rng.normal(0, 2.0, N)
    path_gain_raw = 6 + 3*np.sin(t/230 + 0.8) + rng.normal(0, 1.0, N)
    traffic_raw_s = np.maximum(40, 500 + 230*np.sin(t/95) + rng.normal(0, 100, N))
    delay_raw = np.maximum(1, 8 + 0.012*traffic_raw_s + rng.normal(0, 2, N))
    energy_raw = np.clip(0.45 + 0.18*np.sin(t/300) + rng.normal(0, .07, N), .05, .95)

df = pd.DataFrame({
    "time_index": t,
    "snr_db": snr_raw,
    "path_gain": path_gain_raw,
    "traffic_load": traffic_raw_s,
    "observed_delay_ms": delay_raw,
    "observed_energy": energy_raw,
})
for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    df[c] = df[c].interpolate(limit_direction="both").fillna(df[c].median())

# Normalize inputs and derive a bounded harvested-energy reference target.
df["snr_norm"] = robust_unit(df["snr_db"])
df["gain_norm"] = robust_unit(df["path_gain"])
df["traffic_norm"] = robust_unit(df["traffic_load"])
df["delay_norm"] = robust_unit(df["observed_delay_ms"])
df["energy_level"] = np.clip(df["observed_energy"], 0, 1)
# A bounded energy availability state used for supervised next-step prediction.
harvest_component = 0.15 + 0.55*df["gain_norm"].to_numpy() + 0.20*df["snr_norm"].to_numpy()
consumption_component = 0.12 + 0.35*df["traffic_norm"].to_numpy()
energy_state = np.clip(0.55*df["energy_level"].to_numpy() +
                       0.45*harvest_component - 0.20*consumption_component, 0, 1)
df["energy_target"] = np.roll(energy_state, -1)
df.loc[df.index[-1], "energy_target"] = energy_state[-1]
df.to_csv(os.path.join(OUTPUT_DIR, "integrated_enapc6g_dataset.csv"), index=False)
print(df.head(), "\nShape:", df.shape, "| simulated input:", USING_SIMULATED_INPUT)

## 5. LSTM-based future energy availability prediction

In [ ]:
FEATURES = ["snr_norm", "gain_norm", "traffic_norm", "delay_norm", "energy_level"]
LOOKBACK = 24
X_all = df[FEATURES].to_numpy(dtype=np.float32)
y_all = df["energy_target"].to_numpy(dtype=np.float32)

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_all).astype(np.float32)

def make_sequences(X, y, lookback):
    xs, ys = [], []
    for i in range(lookback, len(X)):
        xs.append(X[i-lookback:i])
        ys.append(y[i])
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)

X_seq, y_seq = make_sequences(X_scaled, y_all, LOOKBACK)
split = int(len(X_seq)*0.80)
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

lstm_model = models.Sequential([
    layers.Input(shape=(LOOKBACK, len(FEATURES))),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.20),
    layers.LSTM(32),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
lstm_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history = lstm_model.fit(
    X_train, y_train, validation_split=0.15, epochs=25, batch_size=128,
    callbacks=[early_stop], verbose=1
)
energy_pred = lstm_model.predict(X_test, verbose=0).ravel()
print("Energy prediction MAE:", mean_absolute_error(y_test, energy_pred))
print("Energy prediction RMSE:", mean_squared_error(y_test, energy_pred)**0.5)
print("Energy prediction R2:", r2_score(y_test, energy_pred))
lstm_model.save(os.path.join(OUTPUT_DIR, "enapc_energy_lstm.keras"))

## 6. SWIPT adaptive power-splitting and time-switching

In [ ]:
def swipt_adaptation(predicted_energy, snr_norm, traffic_norm,
                     min_split=0.10, max_split=0.90):
    """Return power-splitting ratio rho and time-switching fraction tau."""
    e = np.clip(predicted_energy, 0, 1)
    s = np.clip(snr_norm, 0, 1)
    q = np.clip(traffic_norm, 0, 1)
    # Low energy / strong harvest opportunity -> more harvesting.
    rho = np.clip(0.72*(1-e) + 0.18*(1-s) + 0.10*(1-q), min_split, max_split)
    # Fraction of frame reserved for energy harvesting.
    tau = np.clip(0.65*(1-e) + 0.20*(1-s) + 0.15*q, 0.05, 0.75)
    return rho, tau

rho_test, tau_test = swipt_adaptation(
    energy_pred,
    df.iloc[LOOKBACK+split:LOOKBACK+split+len(energy_pred)]["snr_norm"].to_numpy(),
    df.iloc[LOOKBACK+split:LOOKBACK+split+len(energy_pred)]["traffic_norm"].to_numpy()
)
print("Mean power-splitting ratio:", rho_test.mean())
print("Mean time-switching fraction:", tau_test.mean())

## 7. Communication-system model and objective functions

In [ ]:
# Normalized decision vector:
# x[0] transmit power fraction; x[1] bandwidth fraction;
# x[2] harvested-energy allocation; x[3] active scheduling fraction.
LOW = np.array([0.05, 0.10, 0.05, 0.10])
HIGH = np.array([1.00, 1.00, 0.95, 1.00])

SYSTEM = {
    "bandwidth_hz": 20e6,
    "noise_power_w": 1e-9,
    "max_tx_power_w": 1.0,
    "frame_seconds": 0.010,
    "circuit_power_w": 0.08,
    "harvest_efficiency": 0.65,
    "packet_bits": 12000,
}

def evaluate_policy(x, state):
    p_frac, bw_frac, harvest_frac, active_frac = np.clip(x, LOW, HIGH)
    snr_lin = 10**(float(state["snr_db"])/10)
    channel_gain = max(1e-4, float(state["gain_norm"]))
    traffic = max(0.02, float(state["traffic_norm"]))
    e_avail = np.clip(float(state["energy_pred"]), 0, 1)
    rho = np.clip(float(state["rho"]), 0.05, 0.95)
    tau = np.clip(float(state["tau"]), 0.05, 0.75)

    tx_power = p_frac * SYSTEM["max_tx_power_w"]
    bandwidth = bw_frac * SYSTEM["bandwidth_hz"]
    effective_info_fraction = max(0.05, (1-tau)*(1-rho))
    effective_harvest_fraction = min(0.95, tau + (1-tau)*rho)
    effective_snr = tx_power * (0.2 + channel_gain) * snr_lin / SYSTEM["noise_power_w"] * 1e-9
    rate_bps = bandwidth * np.log2(1 + max(0, effective_snr)) * effective_info_fraction
    throughput_mbps = rate_bps / 1e6 * active_frac
    tx_energy = (tx_power + SYSTEM["circuit_power_w"])*SYSTEM["frame_seconds"]*active_frac
    harvested_energy = (SYSTEM["harvest_efficiency"] * tx_power *
                        effective_harvest_fraction * SYSTEM["frame_seconds"])
    available_energy = e_avail * 0.02 + harvested_energy
    energy_deficit = max(0.0, tx_energy - available_energy)
    # Queueing approximation: delay grows with offered load / service capacity.
    service_mbps = max(0.05, throughput_mbps)
    delay_ms = 1.0 + 12.0*traffic/(service_mbps + 0.15) + 20.0*energy_deficit
    reliability = np.clip(1 - np.exp(-service_mbps/(0.25 + 1.5*traffic)), 0, 1)
    energy_neutrality = np.clip((available_energy + 1e-9)/(tx_energy + 1e-9), 0, 2)
    # Objectives are minimized: negative throughput, energy cost, delay.
    return {
        "throughput_mbps": float(throughput_mbps),
        "energy_j": float(tx_energy),
        "harvested_energy_j": float(harvested_energy),
        "delay_ms": float(delay_ms),
        "reliability": float(reliability),
        "energy_neutrality": float(energy_neutrality),
        "energy_deficit_j": float(energy_deficit),
        "objectives": np.array([-throughput_mbps, tx_energy, delay_ms], dtype=float)
    }

def scalar_fitness(x, state, weights=(0.45, 0.25, 0.30)):
    m = evaluate_policy(x, state)
    # Normalize terms to keep scales comparable.
    return (weights[0] * (-m["throughput_mbps"]/100.0) +
            weights[1] * (m["energy_j"]/0.02) +
            weights[2] * (m["delay_ms"]/100.0) +
            2.0*m["energy_deficit_j"])

## 8. Multi-Objective Adaptive Whale Optimization Algorithm (MOAWO)

In [ ]:
def dominates(a, b):
    return np.all(a <= b) and np.any(a < b)

def update_archive(archive, candidate, max_size=40):
    objs = candidate["obj"]
    if any(dominates(item["obj"], objs) for item in archive):
        return archive
    archive = [item for item in archive if not dominates(objs, item["obj"])]
    archive.append(candidate)
    if len(archive) > max_size:
        # Keep a spread of solutions using objective-space crowding distance.
        F = np.array([it["obj"] for it in archive])
        crowd = np.zeros(len(archive))
        for j in range(F.shape[1]):
            order = np.argsort(F[:, j])
            crowd[order[0]] = crowd[order[-1]] = np.inf
            span = F[order[-1], j] - F[order[0], j] + 1e-12
            for k in range(1, len(order)-1):
                crowd[order[k]] += (F[order[k+1], j]-F[order[k-1], j])/span
        keep = np.argsort(crowd)[-max_size:]
        archive = [archive[i] for i in keep]
    return archive

def moawo_optimize(state, population=28, iterations=45, archive_size=40):
    dim = len(LOW)
    whales = rng.uniform(LOW, HIGH, size=(population, dim))
    archive = []
    for x in whales:
        m = evaluate_policy(x, state)
        archive = update_archive(archive, {"x": x.copy(), "obj": m["objectives"]}, archive_size)
    for it in range(iterations):
        a = 2 - 2*it/max(1, iterations-1)
        leader = archive[rng.integers(len(archive))]["x"]
        new_whales = []
        for i, x in enumerate(whales):
            r1, r2 = rng.random(), rng.random()
            A = 2*a*r1 - a
            C = 2*r2
            l = rng.uniform(-1, 1)
            b = 1.0
            p = rng.random()
            if p < 0.5:
                if abs(A) < 1:
                    D = np.abs(C*leader - x)
                    xn = leader - A*D
                else:
                    peer = whales[rng.integers(population)]
                    D = np.abs(C*peer - x)
                    xn = peer - A*D
            else:
                D = np.abs(leader - x)
                xn = D*np.exp(b*l)*np.cos(2*np.pi*l) + leader
            # Adaptive mutation encourages exploration early and refinement later.
            mutation_scale = 0.08*(1-it/iterations) + 0.005
            xn += rng.normal(0, mutation_scale, dim)*(HIGH-LOW)
            xn = np.clip(xn, LOW, HIGH)
            new_whales.append(xn)
            m = evaluate_policy(xn, state)
            archive = update_archive(archive, {"x": xn.copy(), "obj": m["objectives"]}, archive_size)
        whales = np.asarray(new_whales)
    # Select a deployable compromise solution from the Pareto archive.
    objs = np.array([a["obj"] for a in archive])
    mins, maxs = objs.min(axis=0), objs.max(axis=0)
    norm = (objs-mins)/(maxs-mins+1e-12)
    compromise_idx = np.argmin(np.sqrt((norm**2).sum(axis=1)))
    return archive[compromise_idx]["x"], archive

print("MOAWO optimizer ready.")

## 9. Energy-neutral scheduling controller

In [ ]:
def energy_neutral_controller(state, optimized_x, reserve=0.12):
    """Decide whether to transmit, reduce activity, or harvest energy."""
    predicted = float(np.clip(state["energy_pred"], 0, 1))
    x = np.array(optimized_x, dtype=float).copy()
    if predicted < reserve:
        x[3] = min(x[3], 0.20)      # restrict active scheduling
        x[0] = min(x[0], 0.35)      # reduce transmit power
        action = "energy_conservation"
    elif predicted < 0.35:
        x[3] = min(x[3], 0.55)
        x[2] = max(x[2], 0.55)      # allocate more to harvesting
        action = "harvest_priority"
    else:
        action = "normal_transmission"
    return np.clip(x, LOW, HIGH), action

## 10. Run the proposed ENAPC-6G policy over the test trace

In [ ]:
test_start = LOOKBACK + split
test_rows = df.iloc[test_start:test_start+len(energy_pred)].reset_index(drop=True)
records = []
for i, row in test_rows.iterrows():
    ehat = float(energy_pred[i])
    rho, tau = swipt_adaptation(ehat, row["snr_norm"], row["traffic_norm"])
    state = {
        "snr_db": row["snr_db"], "gain_norm": row["gain_norm"],
        "traffic_norm": row["traffic_norm"], "energy_pred": ehat,
        "rho": float(rho), "tau": float(tau)
    }
    x_opt, pareto = moawo_optimize(state, population=20, iterations=30, archive_size=30)
    x_sched, action = energy_neutral_controller(state, x_opt)
    metrics = evaluate_policy(x_sched, state)
    records.append({
        "time_index": int(row["time_index"]), "energy_pred": ehat,
        "rho": rho, "tau": tau, "tx_power_fraction": x_sched[0],
        "bandwidth_fraction": x_sched[1], "harvest_fraction": x_sched[2],
        "active_fraction": x_sched[3], "controller_action": action,
        **{k:v for k,v in metrics.items() if k != "objectives"}
    })
    if (i+1) % 100 == 0: print(f"Processed {i+1}/{len(test_rows)} time slots")

proposed_results = pd.DataFrame(records)
proposed_results.to_csv(os.path.join(OUTPUT_DIR, "ENAPC6G_proposed_results.csv"), index=False)
proposed_results.head()

## 11. Baseline policies and comparative evaluation

In [ ]:
def run_baseline(name, row, energy_hat):
    state = {
        "snr_db": row["snr_db"], "gain_norm": row["gain_norm"],
        "traffic_norm": row["traffic_norm"], "energy_pred": energy_hat,
    }
    if name == "Fixed-power":
        x = np.array([0.75, 0.70, 0.20, 0.90])
    elif name == "Greedy-throughput":
        x = np.array([1.00, 1.00, 0.05, 1.00])
    elif name == "Energy-aware":
        x = np.array([0.45, 0.60, 0.60, 0.55])
    else:
        x = np.array([0.60, 0.65, 0.35, 0.75])
    rho, tau = swipt_adaptation(energy_hat, row["snr_norm"], row["traffic_norm"])
    state["rho"], state["tau"] = float(rho), float(tau)
    if name == "ENAPC-6G":
        x, _ = energy_neutral_controller(state, x)
    return evaluate_policy(x, state)

baseline_rows = []
for i, row in test_rows.iterrows():
    eh = float(energy_pred[i])
    for method in ["Fixed-power", "Greedy-throughput", "Energy-aware"]:
        m = run_baseline(method, row, eh)
        baseline_rows.append({"method": method, **{k:v for k,v in m.items() if k != "objectives"}})
    pm = proposed_results.iloc[i]
    baseline_rows.append({"method": "ENAPC-6G", **{k:pm[k] for k in [
        "throughput_mbps","energy_j","harvested_energy_j","delay_ms",
        "reliability","energy_neutrality","energy_deficit_j"]}})
baseline_df = pd.DataFrame(baseline_rows)
summary = baseline_df.groupby("method").agg({
    "throughput_mbps":"mean", "energy_j":"mean", "harvested_energy_j":"mean",
    "delay_ms":"mean", "reliability":"mean", "energy_neutrality":"mean",
    "energy_deficit_j":"mean"
}).reset_index()
summary.to_csv(os.path.join(OUTPUT_DIR, "baseline_comparison.csv"), index=False)
summary

## 12. Scenario analysis: channel, traffic and harvesting conditions

In [ ]:
def scenario_evaluate(channel_factor=1.0, traffic_factor=1.0, harvest_factor=1.0, n_slots=250):
    out = []
    idxs = np.linspace(0, len(test_rows)-1, min(n_slots, len(test_rows))).astype(int)
    for idx in idxs:
        row = test_rows.iloc[idx]
        eh = float(energy_pred[idx])
        state = {
            "snr_db": float(row["snr_db"] + 10*np.log10(max(channel_factor, 0.05))),
            "gain_norm": float(np.clip(row["gain_norm"]*channel_factor, 0, 1)),
            "traffic_norm": float(np.clip(row["traffic_norm"]*traffic_factor, 0, 1)),
            "energy_pred": float(np.clip(eh*harvest_factor, 0, 1)),
        }
        state["rho"], state["tau"] = swipt_adaptation(
            state["energy_pred"], row["snr_norm"], state["traffic_norm"])
        x, _ = moawo_optimize(state, population=14, iterations=20, archive_size=20)
        x, action = energy_neutral_controller(state, x)
        m = evaluate_policy(x, state)
        out.append({"channel_condition":channel_factor, "traffic_condition":traffic_factor,
                    "harvesting_condition":harvest_factor, "action":action,
                    **{k:v for k,v in m.items() if k!="objectives"}})
    return pd.DataFrame(out)

scenario_specs = [
    ("Poor channel / low load / low harvest", 0.55, 0.45, 0.45),
    ("Average channel / average load / average harvest", 1.0, 1.0, 1.0),
    ("Good channel / high load / high harvest", 1.35, 1.45, 1.35),
    ("Poor channel / high load / low harvest", 0.55, 1.45, 0.45),
]
scenario_frames = []
for label, ch, tr, ha in scenario_specs:
    s = scenario_evaluate(ch, tr, ha, n_slots=150)
    s["scenario"] = label
    scenario_frames.append(s)
scenario_df = pd.concat(scenario_frames, ignore_index=True)
scenario_summary = scenario_df.groupby("scenario").agg({
    "throughput_mbps":"mean", "energy_j":"mean", "delay_ms":"mean",
    "reliability":"mean", "energy_neutrality":"mean", "energy_deficit_j":"mean"
}).reset_index()
scenario_df.to_csv(os.path.join(OUTPUT_DIR, "scenario_slot_results.csv"), index=False)
scenario_summary.to_csv(os.path.join(OUTPUT_DIR, "scenario_summary.csv"), index=False)
scenario_summary

## 13. Performance metrics

In [ ]:
def compute_network_metrics(frame):
    return pd.Series({
        "Mean Throughput (Mbps)": frame["throughput_mbps"].mean(),
        "Mean Energy Consumption (J)": frame["energy_j"].mean(),
        "Mean Harvested Energy (J)": frame["harvested_energy_j"].mean(),
        "Mean Delay (ms)": frame["delay_ms"].mean(),
        "Mean Reliability": frame["reliability"].mean(),
        "Energy-Neutrality Ratio": (frame["energy_neutrality"] >= 1.0).mean(),
        "Energy Deficit Probability": (frame["energy_deficit_j"] > 0).mean(),
        "Mean Energy Deficit (J)": frame["energy_deficit_j"].mean(),
    })

metric_table = pd.DataFrame({
    method: compute_network_metrics(group)
    for method, group in baseline_df.groupby("method")
}).T
metric_table.to_csv(os.path.join(OUTPUT_DIR, "performance_metrics.csv"))
metric_table

## 14. Visualizations

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.figure(figsize=(10,5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("MSE loss"); plt.title("LSTM energy prediction training")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "lstm_training_loss.png"), dpi=300)
plt.show()

plot_cols = ["throughput_mbps", "energy_j", "delay_ms", "reliability", "energy_neutrality"]
for col in plot_cols:
    plt.figure(figsize=(9,5))
    sns.barplot(data=baseline_df, x="method", y=col, errorbar="sd")
    plt.xticks(rotation=15, ha="right")
    plt.title(f"Policy comparison: {col.replace('_',' ').title()}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"comparison_{col}.png"), dpi=300)
    plt.show()

plt.figure(figsize=(11,5))
sns.lineplot(data=proposed_results, x="time_index", y="energy_pred", label="Predicted energy")
plt.xlabel("Time slot"); plt.ylabel("Energy availability (normalized)")
plt.title("Predicted energy availability over test period")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "predicted_energy_trace.png"), dpi=300)
plt.show()

plt.figure(figsize=(10,5))
sns.barplot(data=scenario_summary, x="scenario", y="throughput_mbps")
plt.xticks(rotation=20, ha="right"); plt.ylabel("Mean throughput (Mbps)")
plt.title("ENAPC-6G under channel, traffic and harvesting scenarios")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "scenario_throughput.png"), dpi=300)
plt.show()

## 15. Export Excel workbook and ZIP package

In [ ]:
excel_path = os.path.join(OUTPUT_DIR, "ENAPC6G_complete_results.xlsx")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Integrated_Data", index=False)
    proposed_results.to_excel(writer, sheet_name="ENAPC6G_Proposed", index=False)
    summary.to_excel(writer, sheet_name="Baseline_Comparison", index=False)
    metric_table.to_excel(writer, sheet_name="Performance_Metrics")
    scenario_summary.to_excel(writer, sheet_name="Scenario_Summary", index=False)
    scenario_df.to_excel(writer, sheet_name="Scenario_Details", index=False)

zip_path = os.path.join(OUTPUT_DIR, "ENAPC6G_all_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for file in files:
            if file == os.path.basename(zip_path): continue
            full = os.path.join(root, file)
            zf.write(full, arcname=os.path.relpath(full, OUTPUT_DIR))

print("Saved outputs:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", f)
print("\nExcel:", excel_path)
print("ZIP:", zip_path)

## 16. Interpretation and experimental reporting checklist

- Confirm the exact versions and preprocessing procedures for DeepMIMO, 6G-PATH and 5G traffic datasets.
- Verify that the chosen columns represent compatible units. This notebook aligns extracted records by row index; it does not claim that records from separate datasets are synchronized measurements.
- If the source datasets have timestamps, map them to a common time base before training. If they have scenario, user, base-station or channel identifiers, preserve those identifiers and use group-aware splits to avoid leakage.
- Report whether the run used source datasets or the built-in simulation (`USING_SIMULATED_INPUT`).
- Report LSTM prediction MAE/RMSE/R² separately from network-control metrics.
- MOAWO is implemented as a Pareto-archive whale optimizer with adaptive exploration and a compromise-solution selector. Tune population, iterations and objective weights for the intended deployment scenario.
- The link, queueing, energy-harvesting and reliability equations are tractable simulation models. Calibrate them with the radio/PHY and hardware assumptions of the target 6G system before making deployment claims.
- Energy neutrality is reported as the fraction of evaluated slots with harvested-plus-available energy at least equal to modeled consumption.